# Alfa-Code Tiny-50M — Colab Free T4
Runtime → Change runtime type → T4 GPU. Then Run all.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers datasets tokenizers accelerate huggingface_hub pyyaml bitsandbytes

In [ ]:
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not visible to torch. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session and Run all.')


In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = None
if tok:
    login(token=tok)
    print('logged in')
else:
    print('No HF_TOKEN secret — Hub upload step will be skipped. Add it via the key icon if needed.')


In [ ]:
# Code comes from public GitHub (no token needed) — avoids 401 from private HF repo
%cd /content
!rm -rf /content/alfa
!git clone https://github.com/mrsandipmandal/alfa-code.git /content/alfa
%cd /content/alfa
!ls scripts/ configs/

In [ ]:
# Pick model ONCE here — every cell below follows it.
MODEL = "base-1B"  # or "tiny-50M" for the fast T4-safe test
if MODEL == "base-1B":
    CONFIG, OUT, EPOCHS, SEQ_LEN, BS, ROWS, CHAT_ROWS = (
        "configs/base-1B.yaml", "/content/outputs/base-1B", 3, 1024, 1, 5000, 2000)
else:
    CONFIG, OUT, EPOCHS, SEQ_LEN, BS, ROWS, CHAT_ROWS = (
        "configs/tiny-50M.yaml", "/content/outputs/tiny-50M", 1, 2048, 2, 1500, 500)
print("MODEL:", MODEL, "| OUT:", OUT, "| epochs:", EPOCHS, "| seq:", SEQ_LEN, "| rows:", ROWS, "| chat:", CHAT_ROWS)


In [ ]:
!python scripts/download_data.py --dataset multimodal_mix --max-rows {ROWS} --overwrite
!python scripts/download_data.py --dataset chat_qa --max-rows {CHAT_ROWS}
!python scripts/train_tokenizer.py
!ls -lh tokenizers/ data/processed/


In [ ]:
!python scripts/train_hf.py --data data/processed/train.jsonl --out {OUT} --epochs {EPOCHS} --config {CONFIG} --max-seq-len {SEQ_LEN} --batch-size {BS}
!ls -lh {OUT}


In [ ]:
# SELF-LEARNING CYCLE (Colab manual): generate -> filter -> append -> retrain.
# Bump CYCLE every round: 1, 2, 3... Review data before promoting.
CYCLE = 1
!python scripts/self_learn.py --cycle {CYCLE} --topics data/seeds/topics.txt --from-train 100 --n 200 --max-tokens 256 --append
# retrain on merged data into a VERSIONED folder (does NOT overwrite your model):
!python scripts/train_hf.py --data data/processed/train.jsonl --out /content/outputs/self-cycle-{CYCLE} --epochs 1 --config {CONFIG} --max-seq-len {SEQ_LEN} --batch-size {BS}
!ls -lh /content/outputs/self-cycle-{CYCLE}
# review (infer sample + cycle report), then promote when happy:
#   !cp -r /content/outputs/self-cycle-{CYCLE} {OUT}
# then run the upload cell below.


In [ ]:
# Push back to Hub (uploads every trained model found: tiny-50M, base-1B, self-cycle-N).
# Needs HF_TOKEN in Colab Secrets: key icon on the left.
import os
from pathlib import Path
tok = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = os.getenv('HF_TOKEN')
if not tok:
    print('SKIP: no HF_TOKEN secret set.')
else:
    from huggingface_hub import HfApi
    api = HfApi(token=tok)
    found = sorted(p for p in Path('/content/outputs').glob('*') if (p / 'config.json').exists())
    if not found:
        print('nothing trained yet in /content/outputs')
    for local in found:
        api.upload_folder(folder_path=str(local), repo_id='mrsandip/Alfa-Code', path_in_repo=f'outputs/{local.name}')
        print('uploaded', local.name)


In [ ]:
# Launch UI: pulls latest code, then serves the newest trained model below (else mock).
# Open the gradio.live link that appears below.
%cd /content/alfa
!git pull origin main
import os
from pathlib import Path
for _cand in ['/content/outputs/base-1B', '/content/outputs/tiny-50M']:
    if (Path(_cand) / 'config.json').exists():
        os.environ['ALFA_MODEL'] = _cand
        print('using model:', _cand)
        break
import importlib, app.model_backend as _mb, app.ui as _ui
importlib.reload(_mb); importlib.reload(_ui)
_ui.launch_share()
